# Gabarito

Responde as perguntas do dataset de avaliação de forma **determinística** (pandas), gerando a resposta esperada para depois avaliar o RAG (`src/Evals.ipynb`).

Cada pergunta testa um cenário: `consulta` (busca exata/BM25), `semantica` (busca densa), `fora_escopo` (recusa), `analitica` (tools) e `qualidade` (relatório de qualidade).

> **ADAPTE:** troque as perguntas pelas da sua base. Mantenha poucas e bem escolhidas: cada uma custa 1 ou 2 requisições na geração e cerca de 2 no juiz do RAGAS.

In [1]:
import pandas as pd

categorias = pd.read_csv('../data/processed/categorias.csv')
clientes = pd.read_csv('../data/processed/clientes.csv')
funcionarios = pd.read_csv('../data/processed/funcionarios.csv')
itens = pd.read_csv('../data/processed/itens_pedido.csv')
pedidos = pd.read_csv('../data/processed/pedidos.csv')
produtos = pd.read_csv('../data/processed/produtos.csv')
transportadoras = pd.read_csv('../data/processed/transportadoras.csv')
qualidade = pd.read_csv('../data/quality/qualidade_dados.csv')

NOT_FOUND = 'Não encontrei essa informação nos documentos disponíveis.'

perguntas = []  # cada pergunta vira um dicionário; no final a lista vira o dataset.csv


def adicionar(id_, tipo, pergunta, resposta, ids='', tool='', valores=''):
    # ids e valores: vários itens separados por ';' (ex.: 'funcionarios:1;funcionarios:8')
    perguntas.append({'id': id_, 'tipo': tipo, 'pergunta': pergunta, 'resposta_esperada': resposta,
                      'ids_esperados': ids, 'tool_esperada': tool, 'valores_esperados': valores})


def dolares(valor):
    return f'US$ {valor:,.2f}'

> **Colunas do dataset:** `ids_esperados` alimenta o `context_recall` e `valores_esperados` alimenta o `values_recall` (números que precisam aparecer na resposta).

#### Consulta
Testam a busca exata (BM25): nomes e IDs que aparecem literalmente nos documentos.

P01 - Qual o preço unitário do Côte de Blaye?

*Consulta simples por nome de produto (com acento: testa a normalização do `tokenize`).*

In [2]:
produto = produtos[produtos['produto'] == 'Côte de Blaye'].iloc[0]
print(produto[['id_produto', 'produto', 'preco_unitario']], '\n')

resposta = f"{dolares(produto['preco_unitario'])} (Côte de Blaye, produto {produto['id_produto']})."
adicionar('P01', 'consulta', 'Qual o preço unitário do Côte de Blaye?', resposta,
          ids=f"produtos:{produto['id_produto']}", valores=str(produto['preco_unitario']))

id_produto                   38
produto           Côte de Blaye
preco_unitario            263.5
Name: 37, dtype: object 



P02 - Quais são os detalhes do pedido 10248?

*Busca por ID exato. O documento do pedido só é autoexplicativo por causa do `enrich()` do `ingest.py`: cliente, funcionário, transportadora e itens vêm de outras tabelas.*

In [3]:
pedido = pedidos[pedidos['id_pedido'] == 10248].iloc[0]
cliente = clientes[clientes['id_cliente'] == pedido['id_cliente']]['empresa'].iloc[0]
funcionario = funcionarios[funcionarios['id_funcionario'] == pedido['id_funcionario']]['nome'].iloc[0]
transportadora = transportadoras[transportadoras['id_transportadora'] == pedido['id_transportadora']]['empresa'].iloc[0]
itens_pedido = itens[itens['id_pedido'] == 10248].merge(produtos[['id_produto', 'produto']], on='id_produto')
print(pedido, '\n')
print(itens_pedido[['produto', 'quantidade', 'preco_unitario']], '\n')

lista_itens = ', '.join(f"{r.produto} ({r.quantidade} un.)" for r in itens_pedido.itertuples())
resposta = (f"Pedido de {pedido['data_pedido']} do cliente {cliente} ({pedido['id_cliente']}), atendido por "
            f"{funcionario}; enviado em {pedido['data_envio']} pela {transportadora}, frete de {dolares(pedido['frete'])}, "
            f"destino {pedido['cidade_entrega']}/{pedido['pais_entrega']}. Itens: {lista_itens}. "
            f"Valor total: {dolares(pedido['valor_total'])}.")
adicionar('P02', 'consulta', 'Quais são os detalhes do pedido 10248?', resposta,
          ids='pedidos:10248', valores=f"{pedido['valor_total']};{pedido['frete']}")

id_pedido                                10248
id_cliente                               VINET
id_funcionario                               5
data_pedido                         1996-07-04
data_prevista                       1996-08-01
data_envio                          1996-07-16
id_transportadora                            3
frete                                    32.38
destinatario         Vins et alcools Chevalier
endereco_entrega            59 rue de l'Abbaye
cidade_entrega                           Reims
regiao_entrega                             NaN
cep_entrega                              51100
pais_entrega                            France
valor_total                              440.0
n_itens                                      3
ano_mes                                1996-07
enviado                                   True
atrasado                                 False
Name: 0, dtype: object 

                         produto  quantidade  preco_unitario
0                 Que

P03 - Quem é o supervisor da Anne Dodsworth?

*Relação entre linhas da mesma tabela: só funciona porque o `enrich()` copia `nome_supervisor` para o documento.*

In [4]:
anne = funcionarios[funcionarios['nome'] == 'Anne Dodsworth'].iloc[0]
supervisor = funcionarios[funcionarios['id_funcionario'] == anne['id_supervisor']].iloc[0]
print(anne[['id_funcionario', 'nome', 'id_supervisor']], '\n')

resposta = f"{supervisor['nome']} ({supervisor['cargo']}, funcionário {supervisor['id_funcionario']})."
adicionar('P03', 'consulta', 'Quem é o supervisor da Anne Dodsworth?', resposta,
          ids=f"funcionarios:{anne['id_funcionario']}")

id_funcionario                 9
nome              Anne Dodsworth
id_supervisor                5.0
Name: 8, dtype: object 



#### Semantica
Testam a busca densa: a pergunta (em português) usa palavras diferentes das do documento (em inglês), então o BM25 sozinho não resolve.

P04 - Quais funcionários estudaram psicologia?

*"psicologia" ≈ "psychology" no campo `observacoes`. Dois documentos esperados.*

In [5]:
psicologia = funcionarios[funcionarios['observacoes'].str.contains('psychology', case=False)]
print(psicologia[['id_funcionario', 'nome', 'observacoes']], '\n')

resposta = ' e '.join(f"{r.nome} (funcionário {r.id_funcionario})" for r in psicologia.itertuples()) + \
           ', segundo as observações do cadastro.'
adicionar('P04', 'semantica', 'Quais funcionários estudaram psicologia?', resposta,
          ids=';'.join(f'funcionarios:{i}' for i in psicologia['id_funcionario']))

   id_funcionario            nome  \
0               1   Nancy Davolio   
7               8  Laura Callahan   

                                         observacoes  
0  Education includes a BA in psychology from Col...  
7  Laura received a BA in psychology from the Uni...   



P05 - Em qual categoria ficam os queijos?

*"queijos" ≈ "Cheeses" na descrição da categoria.*

In [6]:
categoria = categorias[categorias['descricao'].str.contains('Cheese')].iloc[0]
print(categoria, '\n')

resposta = f"{categoria['categoria']} (categoria {categoria['id_categoria']}), descrita como \"{categoria['descricao']}\"."
adicionar('P05', 'semantica', 'Em qual categoria ficam os queijos?', resposta,
          ids=f"categorias:{categoria['id_categoria']}")

id_categoria                 4
categoria       Dairy Products
descricao              Cheeses
Name: 3, dtype: object 



#### Fora de escopo
A resposta esperada é sempre a frase de recusa do `generate.py`. O código confirma que o assunto não aparece na base.

In [7]:
texto_base = ''
for tabela in [categorias, clientes, funcionarios, pedidos, produtos, transportadoras, qualidade]:
    texto_base += tabela.to_string().lower()

P06 - Qual a capital da França?

*Totalmente fora do tema: o threshold deve barrar antes de chamar a LLM. Note que "France" existe na base (país de clientes), o que torna o teste mais difícil para o BM25.*

In [8]:
print('capital na base?', 'capital' in texto_base)
adicionar('P06', 'fora_escopo', 'Qual a capital da França?', NOT_FOUND)

capital na base? True


P07 - Qual o salário da Nancy Davolio?

*Mesmo domínio e com um nome que existe na base: tende a passar no threshold e precisa ser recusada pelo prompt.*

In [9]:
print('Colunas de funcionários:', list(funcionarios.columns))
print('salary/salário na base?', 'salary' in texto_base or 'salário' in texto_base)
adicionar('P07', 'fora_escopo', 'Qual o salário da Nancy Davolio?', NOT_FOUND)

Colunas de funcionários: ['id_funcionario', 'nome', 'cargo', 'tratamento', 'data_nascimento', 'data_admissao', 'endereco', 'cidade', 'regiao', 'cep', 'pais', 'telefone', 'ramal', 'observacoes', 'id_supervisor']
salary/salário na base? False


#### Analitica
Calculadas com pandas puro, **sem importar o `tools.py`**: se o gabarito usasse o mesmo código das tools, um erro de cálculo apareceria igual nos dois e passaria na avaliação.

Faturamento = soma de preço do item × quantidade × (1 − desconto), sem frete; volume = soma de quantidade.

In [10]:
base = itens.merge(pedidos[['id_pedido', 'id_cliente', 'id_funcionario', 'data_pedido', 'pais_entrega']], on='id_pedido')
base['valor'] = base['preco_unitario'] * base['quantidade'] * (1 - base['desconto'])
base['ano'] = base['data_pedido'].str[:4]
print('Itens:', len(base), '| período:', base['data_pedido'].min(), 'a', base['data_pedido'].max())

Itens: 2155 | período: 1996-07-04 a 1998-05-06


P08 - Qual funcionário teve o melhor desempenho em vendas?

*Ranking por faturamento no período todo. A resposta precisa trazer o valor e o período.*

In [11]:
faturamento = base.groupby('id_funcionario')['valor'].sum().sort_values(ascending=False)
print(faturamento.head(3), '\n')

id_melhor, valor_melhor = faturamento.index[0], round(faturamento.iloc[0], 2)
nome_melhor = funcionarios[funcionarios['id_funcionario'] == id_melhor]['nome'].iloc[0]
resposta = (f"{nome_melhor} (funcionário {id_melhor}), com faturamento de {dolares(valor_melhor)} entre "
            f"{base['data_pedido'].min()[:7]} e {base['data_pedido'].max()[:7]}.")
adicionar('P08', 'analitica', 'Qual funcionário teve o melhor desempenho em vendas?', resposta,
          tool='ranking', valores=str(valor_melhor))

id_funcionario
4    232890.8460
3    202812.8430
1    192107.6045
Name: valor, dtype: float64 



P09 - Quem são os top 3 clientes por volume?

*Volume = unidades, e não faturamento: o top 3 por faturamento é diferente (QUICK-Stop lidera).*

In [12]:
volume = base.groupby('id_cliente')['quantidade'].sum().sort_values(ascending=False).head(3)
print(volume, '\n')

partes, valores = [], []
for posicao, (id_cliente, unidades) in enumerate(volume.items(), start=1):
    nome = clientes[clientes['id_cliente'] == id_cliente]['empresa'].iloc[0]
    partes.append(f"{posicao}. {nome} ({id_cliente}): {unidades} unidades")
    valores.append(str(unidades))

resposta = '; '.join(partes) + '. Volume = soma das unidades dos itens de pedido.'
adicionar('P09', 'analitica', 'Quem são os top 3 clientes por volume?', resposta,
          tool='ranking', valores=';'.join(valores))

id_cliente
SAVEA    4958
ERNSH    4543
QUICK    3961
Name: quantidade, dtype: int64 



P10 - O faturamento da Alemanha caiu de 1997 para 1998?

*Pegadinha de propósito: os números mostram queda, mas 1998 só tem pedidos até maio. A resposta esperada exige a ressalva do ano parcial.*

In [13]:
alemanha = base[base['pais_entrega'] == 'Germany'].groupby('ano')['valor'].sum().round(2)
meses_1998 = sorted(pedidos[pedidos['data_pedido'].str[:4] == '1998']['data_pedido'].str[:7].unique())
print(alemanha, '\n', 'meses de 1998:', meses_1998, '\n')

antes, depois = alemanha['1997'], alemanha['1998']
variacao = round((depois / antes - 1) * 100, 2)
resposta = (f"Os valores caem de {dolares(antes)} em 1997 para {dolares(depois)} em 1998 ({variacao}%), mas 1998 é um "
            f"ano parcial: só há pedidos de {meses_1998[0]} a {meses_1998[-1]} ({len(meses_1998)} meses). "
            "A comparação com o ano cheio de 1997 não indica queda real.")
adicionar('P10', 'analitica', 'O faturamento da Alemanha caiu de 1997 para 1998?', resposta,
          tool='comparar_anos', valores=f'{antes};{depois}')

ano
1996     35407.15
1997    117320.16
1998     77557.32
Name: valor, dtype: float64 
 meses de 1998: ['1998-01', '1998-02', '1998-03', '1998-04', '1998-05'] 



P11 - Quanto a Margaret Peacock faturou em 1997?

*Cobre a tool `consultar_total` com dois filtros (funcionário por nome e ano cheio).*

In [14]:
id_margaret = funcionarios[funcionarios['nome'] == 'Margaret Peacock']['id_funcionario'].iloc[0]
vendas_margaret = base[(base['id_funcionario'] == id_margaret) & (base['ano'] == '1997')]
total = round(vendas_margaret['valor'].sum(), 2)

resposta = (f"{dolares(total)} em {vendas_margaret['id_pedido'].nunique()} pedidos de 1997 "
            f"(Margaret Peacock, funcionário {id_margaret}).")
adicionar('P11', 'analitica', 'Quanto a Margaret Peacock faturou em 1997?', resposta,
          tool='consultar_total', valores=str(total))

#### Qualidade dos dados
Respondidas pelo relatório de qualidade indexado (`data/quality/qualidade_dados.csv`). A resposta esperada usa os próprios textos do relatório.

P12 - Os dados estão prontos para uso?

In [15]:
problema = qualidade[qualidade['id_problema'] == 'Q000'].iloc[0]
resposta = f"{problema['encontrado']} {problema['tratamento']}"
adicionar('P12', 'qualidade', 'Os dados estão prontos para uso?', resposta,
          ids='qualidade_dados:Q000', valores=str(problema['n_registros']))

P13 - Havia problemas de formatação nos arquivos originais?

In [16]:
problema = qualidade[qualidade['id_problema'] == 'Q001'].iloc[0]
resposta = f"Sim. {problema['encontrado']} {problema['tratamento']}"
adicionar('P13', 'qualidade', 'Havia problemas de formatação nos arquivos originais?', resposta,
          ids='qualidade_dados:Q001', valores=str(problema['n_registros']))

## Exportar perguntas e respostas para dataset.csv

In [17]:
dataset = pd.DataFrame(perguntas)
# Se uma célula for executada duas vezes, a pergunta entra repetida: fica só a última versão
dataset = dataset.drop_duplicates(subset='id', keep='last').sort_values('id')

dataset.to_csv('dataset.csv', index=False)  # eval/dataset.csv, o caminho que o generate.py lê
dataset

,id,tipo,pergunta,resposta_esperada,ids_esperados,tool_esperada,valores_esperados
0,P01,consulta,Qual o preço unitário do Côte de Blaye?,"US$ 263.50 (Côte de Blaye, produto 38).",produtos:38,,263.5
1,P02,consulta,Quais são os detalhes do pedido 10248?,Pedido de 1996-07-04 do cliente Vins et alcool...,pedidos:10248,,440.0;32.38
2,P03,consulta,Quem é o supervisor da Anne Dodsworth?,"Steven Buchanan (Sales Manager, funcionário 5).",funcionarios:9,,
3,P04,semantica,Quais funcionários estudaram psicologia?,Nancy Davolio (funcionário 1) e Laura Callahan...,funcionarios:1;funcionarios:8,,
4,P05,semantica,Em qual categoria ficam os queijos?,"Dairy Products (categoria 4), descrita como ""C...",categorias:4,,
5,P06,fora_escopo,Qual a capital da França?,Não encontrei essa informação nos documentos d...,,,
6,P07,fora_escopo,Qual o salário da Nancy Davolio?,Não encontrei essa informação nos documentos d...,,,
7,P08,analitica,Qual funcionário teve o melhor desempenho em v...,"Margaret Peacock (funcionário 4), com faturame...",,ranking,232890.85
8,P09,analitica,Quem são os top 3 clientes por volume?,1. Save-a-lot Markets (SAVEA): 4958 unidades; ...,,ranking,4958;4543;3961
9,P10,analitica,O faturamento da Alemanha caiu de 1997 para 1998?,"Os valores caem de US$ 117,320.16 em 1997 para...",,comparar_anos,117320.16;77557.32
